# Task 5 — offline ranking diagnostics after the 25% probe

The user-reported overall score improved from **0.72700** (v6, about 5% alerts) to **0.74446** (v8, about 15%) and **0.74593** (v9, about 25%). The incremental gain fell from **0.01746** to **0.00147**. This notebook spends no Kaggle submission. It compares saved, already fitted detector rankings on the provided unlabeled Task 5 file to identify a distinct next hypothesis.

All analyses are descriptive: no labels are available, so score overlap, security flags, and feature profiles cannot estimate F3 or choose a fraud detector. The user reports lecturer permission for unsupervised fitting on this Test-only task; the written competition rule differs. No new fitting or saved model is performed here.

In [1]:
from pathlib import Path
import sys,json,joblib
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src'/'pipeline'/'save_data.py').exists())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
SRC=ROOT/'src'
from src.pipeline.random_seed import set_seed
SEED=set_seed()
import numpy as np,pandas as pd
from scipy.stats import spearmanr
from IPython.display import display
DATA_NAME='eda_1_v2'
card=json.loads((SRC/'task5'/'data'/'eda'/f'{DATA_NAME}.json').read_text(encoding='utf8'))
assert card['name']==DATA_NAME and set(card['splits'])=={'test'}
path=SRC/card['splits']['test']['path']
test=pd.read_parquet(path) if card['format']=='parquet' else pd.read_csv(path)
assert test.player_id.is_unique and 'is_anomaly' not in test
X=test.drop(columns='player_id')
v9=pd.read_csv(SRC/'submission'/'kaggle_t12345_t5_alert25_v9.csv')
assert len(test)==len(v9)==25889 and set(v9.task5)=={0,1}
assert v9.task5.sum()==6531
print('Test-only rows:',len(X),'predictors:',X.shape[1])

Test-only rows: 25889 predictors: 213


## 1. Compare saved detector rankings at the same 25% budget

Load the engineered-feature models from their JSON cards. Use a fixed top-25% count for comparison so disagreement reflects ranking, not different saved 5% thresholds. Include each family's previously recorded overlap-fit stability, which is only a repeatability proxy.

In [2]:
names={
    'ocsvm':'t5_r2_ocsvm_nu10_engineered_v2',
    'gmm':'t5_r2_gmm_engineered_v2',
    'iforest':'t5_r2_iforest_ms512_engineered_v2',
    'tail':'t5_r2_tail_engineered_v2',
    'lof':'t5_r2_lof_engineered_v2',
    'pca':'t5_r2_pca_engineered_v2',
}
N=len(X);K=round(.25*N)
scores={};top={};models={};proxy={}
for family,name in names.items():
    model_card=json.loads((SRC/'task5'/'save'/f'{name}.json').read_text(encoding='utf8'))
    assert model_card['data_name']==DATA_NAME
    model=joblib.load(SRC/model_card['model_path'])
    score=np.asarray(model.anomaly_score(X),dtype=float)
    assert score.shape==(N,) and np.isfinite(score).all()
    models[family]=model;scores[family]=score
    top[family]=set(np.argpartition(score,-K)[-K:])
    evaluation=json.loads((SRC/'task5'/'save'/f'{name}-evaluation.json').read_text(encoding='utf8'))
    assert evaluation['metric']=='top5_rank_stability_jaccard'
    proxy[family]=evaluation['score']
assert np.array_equal(models['ocsvm'].predict(X),pd.read_csv(SRC/'submission'/'kaggle_t12345_gate_resnet_v6.csv').task5.to_numpy())
flags=X[['password_changed','vpn_usage','suspicious_login_time','mass_item_sale']].fillna(0).gt(0).any(axis=1).to_numpy()
rows=[]
for family in names:
    mask=np.zeros(N,dtype=bool);mask[list(top[family])]=True
    intersection=len(top[family]&top['ocsvm']);union=len(top[family]|top['ocsvm'])
    rows.append({'family':family,'prior_top5_stability_proxy':proxy[family],
        'spearman_vs_ocsvm':spearmanr(scores['ocsvm'],scores[family]).statistic,
        'top25_jaccard_vs_ocsvm':intersection/union,
        'top25_overlap_count':intersection,
        'top25_security_flag_rate':flags[mask].mean()})
comparison=pd.DataFrame(rows).sort_values('top25_jaccard_vs_ocsvm',ascending=False)
display(comparison.round(3))
print('Fixed top-25% count:',K,'; submitted v9 alert count:',int(v9.task5.sum()))

,family,prior_top5_stability_proxy,spearman_vs_ocsvm,top25_jaccard_vs_ocsvm,top25_overlap_count,top25_security_flag_rate
0,ocsvm,1.000,1.000,1.000,6472,0.313
1,gmm,0.980,0.927,0.698,5321,0.417
4,lof,0.860,0.816,0.628,4994,0.297
5,pca,0.878,0.793,0.539,4535,0.312
2,iforest,0.905,0.755,0.496,4291,0.443
3,tail,0.961,0.604,0.381,3572,0.292


Fixed top-25% count: 6472 ; submitted v9 alert count: 6531


## 2. Separate agreement from model-specific alerts

For the OCSVM and the highest-stability different-family model (GMM), inspect accounts selected by both, by only one, and by neither. Security flags and summary features describe the groups but do not validate them.

In [3]:
a=np.zeros(N,dtype=bool);a[list(top['ocsvm'])]=True
b=np.zeros(N,dtype=bool);b[list(top['gmm'])]=True
groups={'both':a&b,'ocsvm_only':a&~b,'gmm_only':~a&b,'neither':~a&~b}
profile_cols=['security_flag_count','login_move_max_km','location_changes_mean4',
    'device_count_mean4','ip_hash_entropy_mean4','login_time_variance_mean4',
    'purchase_amount_mean_mean4','currency_transfers_mean4',
    'communication_frequency_mean4','missing_feature_count']
assert set(profile_cols)<=set(X)
summary=[]
for group,mask in groups.items():
    summary.append({'group':group,'rows':int(mask.sum()),
        'security_flag_rate':float(flags[mask].mean()),
        'mean_security_flag_count':float(X.loc[mask,'security_flag_count'].mean()),
        'median_missing_count':float(X.loc[mask,'missing_feature_count'].median()),
        'median_login_move_km':float(X.loc[mask,'login_move_max_km'].median())})
display(pd.DataFrame(summary).round(3))
profile=pd.DataFrame({group:X.loc[mask,profile_cols].median(numeric_only=True) for group,mask in groups.items()})
display(profile.round(3))
print('OCSVM/GMM top-25% Jaccard:',len(top['ocsvm']&top['gmm'])/len(top['ocsvm']|top['gmm']))

,group,rows,security_flag_rate,mean_security_flag_count,median_missing_count,median_login_move_km
0,both,5321,0.354,0.434,6.0,122.841
1,ocsvm_only,1151,0.120,0.120,6.0,113.473
2,gmm_only,1151,0.705,0.858,6.0,120.577
3,neither,18266,0.205,0.214,6.0,114.677


,both,ocsvm_only,gmm_only,neither
security_flag_count,0.000,0.000,1.000,0.000
login_move_max_km,122.841,113.473,120.577,114.677
location_changes_mean4,1.879,1.537,1.658,1.412
device_count_mean4,1.667,1.750,1.667,1.667
ip_hash_entropy_mean4,0.550,0.564,0.544,0.549
login_time_variance_mean4,5.163,5.247,5.077,4.867
purchase_amount_mean_mean4,290.217,269.794,271.442,243.841
currency_transfers_mean4,5.898,4.812,5.635,4.793
communication_frequency_mean4,0.766,0.736,0.775,0.740
missing_feature_count,6.000,6.000,6.000,6.000


OCSVM/GMM top-25% Jaccard: 0.6980191525646071


## 3. Which supplied security flags drive the difference?

Break the four supplied binary flags out by group. These are input features and possible risk cues, not confirmed anomaly labels.

In [4]:
flag_cols=['password_changed','vpn_usage','suspicious_login_time','mass_item_sale']
flag_rows=[]
for group,mask in groups.items():
    row={'group':group,'rows':int(mask.sum())}
    for col in flag_cols:
        row[col]=float(X.loc[mask,col].fillna(0).gt(0).mean())
    flag_rows.append(row)
flag_breakdown=pd.DataFrame(flag_rows)
display(flag_breakdown.round(3))
print('GMM-only versus OCSVM-only any-flag rate difference:',
      float(flags[groups['gmm_only']].mean()-flags[groups['ocsvm_only']].mean()))

,group,rows,password_changed,vpn_usage,suspicious_login_time,mass_item_sale
0,both,5321,0.110,0.197,0.064,0.063
1,ocsvm_only,1151,0.003,0.113,0.004,0.000
2,gmm_only,1151,0.208,0.215,0.210,0.225
3,neither,18266,0.035,0.136,0.036,0.006


GMM-only versus OCSVM-only any-flag rate difference: 0.5847089487402258


## 4. Exact submitted v9 versus v11 decisions

The fixed top-25% analysis above uses 6,472 rows per detector. The submitted files each contain 6,531 alerts because v9 used a threshold fitted on its earlier unsupervised sample. Compare the **actual CSV predictions** directly. The close overall scores (v9 0.74593, v11 0.74521) can indicate similar aggregate contribution from swapped accounts; without hidden labels they cannot establish whether the large shared-positive group is correct or whether the shared-negative group hides missed anomalies.

In [5]:
v11=pd.read_csv(SRC/'submission'/'kaggle_t12345_t5_gmm_match_v9_v11.csv')
assert v9[['id','task1','task2','task3','task4']].equals(v11[['id','task1','task2','task3','task4']])
a=v9.task5.to_numpy(dtype=int);b=v11.task5.to_numpy(dtype=int)
assert set(a)==set(b)=={0,1} and a.sum()==b.sum()==6531
actual_table=pd.crosstab(a,b,rownames=['v9'],colnames=['v11'])
display(actual_table)
both_positive=int(((a==1)&(b==1)).sum())
both_negative=int(((a==0)&(b==0)).sum())
v9_only=int(((a==1)&(b==0)).sum())
v11_only=int(((a==0)&(b==1)).sum())
assert (both_positive,both_negative,v9_only,v11_only)==(5377,18204,1154,1154)
print('Agreement all rows:',(a==b).mean())
print('Disagreement rows:',int((a!=b).sum()),'fraction:',(a!=b).mean())
print('Shared alerts as fraction of either model alerts:',both_positive/a.sum())
print('Positive-set Jaccard:',both_positive/(both_positive+v9_only+v11_only))
print('Swapped fraction of each model alert budget:',v9_only/a.sum())
print('User-reported overall score difference, v11 minus v9:',0.74521-0.74593)

v11,0,1
v9,,
0,18204,1154
1,1154,5377


Agreement all rows: 0.9108501680250299
Disagreement rows: 2308 fraction: 0.08914983197497006
Shared alerts as fraction of either model alerts: 0.8233042413106721
Positive-set Jaccard: 0.6996746909564086
Swapped fraction of each model alert budget: 0.17669575868932783
User-reported overall score difference, v11 minus v9: -0.0007199999999999429


## 5. Decision after the offline check

Record a next leaderboard hypothesis only after inspecting the overlap and group profiles. The remaining Kaggle slot need not be used now. A detector-family submission would keep v6 Tasks 1–4 and the confirmed Task 5 label orientation fixed, but it should wait for a fresh submission window or deliberate choice. Keep the 35% cutoff draft as an unscored option.